# Threat Identification Agent Tutorial

The production workflow consists of 4 steps, each calling one agent

| Agent | Input | Adds to threats.json |
|-------|-------|---------------------|
| Threat Identifier | diagram + context | stride_category, element, threat, attack_method |
| Risk Assessor | context + CloudFormation | impact, likelihood |
| Mitigation Planner | (reads threats.json directly) | all_possible_mitigations |
| Mitigation Auditor | CloudFormation + live AWS | mitigations_already_in_place, mitigations_missing, ai_proposed_mitigations, remaining_risk |

This notebook walks you through building the **Threat Identifier** step of a STRIDE-based threat modelling workflow using the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python).

By the end, you will have built an agent that:
- Reads an architecture diagram and business context
- Analyses the system using STRIDE methodology
- Writes the identified threats to a shared `outputs/threats.json` file via an MCP filesystem server
- Validates its own output and retries if validation fails

The patterns in this notebook mirror the production code in `workflow_steps/threat_identification.py`.

## Prerequisites

The project uses `uv` for dependency management. Run this notebook from the project's virtual environment where all packages are already installed.
You do this by selecting the project's python environment as the kernel.

### Imports

In [36]:
import os
import json
import shutil
from pathlib import Path
from datetime import date
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, RunConfig, trace
from agents import OpenAIChatCompletionsModel
from agents.mcp import MCPServerStdio
from agents.tracing import set_trace_processors, TracingProcessor, Span, Trace

### Prepare threats.json file

Copy the content of threats_before_threat_identifier.json into threats.json. It should contain no threats, only metadata.

In [37]:
source = Path.cwd() / "outputs" / "threats_before_threat_identifier.json"
destination = Path.cwd() / "outputs" / "threats.json"

shutil.copy(source, destination)
print(f"Copied {source} → {destination}")

Copied /Users/stania.svobodova/my_projects/threat-modelling-agentic-worklfow/4_without_coordinator/tutorials/outputs/threats_before_threat_identifier.json → /Users/stania.svobodova/my_projects/threat-modelling-agentic-worklfow/4_without_coordinator/tutorials/outputs/threats.json


### Read input files
Prepare files that are used in user prompt. Context and diagram are provided for this tutorial.

In [41]:
INPUTS_DIR = Path.cwd() / "inputs"


def read_input(filename: str) -> str:
    """Read an input file, returning empty string if it doesn't exist."""
    path = INPUTS_DIR / filename
    if not path.exists():
        return ""
    return path.read_text(encoding="utf-8")


context = read_input("context.md")
diagram = read_input("mermaid.md")

## Setup

The Agents SDK talks to any OpenAI-compatible API. In the production workflow we use **LiteLLM** as a proxy, which means we set `LITELLM_API_BASE_URL` and `LITELLM_API_KEY` instead of the default `OPENAI_API_KEY`.

`AsyncOpenAI` is the async HTTP client that the SDK uses under the hood. We create it explicitly here so we can point it at the LiteLLM base URL — this is the "bring your own model" pattern supported by `OpenAIChatCompletionsModel` (used in a later cell).

In [42]:
load_dotenv()

client = AsyncOpenAI(
    base_url=os.getenv("LITELLM_API_BASE_URL"),
    api_key=os.getenv("LITELLM_API_KEY"),
    timeout=300.0,
)

MODEL = "openai/gpt-4o-mini"  # model name as understood by the LiteLLM proxy
MAX_RETRIES = 2
TODAY = date.today().isoformat()

## Agent Instructions

The `instructions` parameter is the agent's **system prompt**. It defines the agent's role, the task it must perform, the format of the output, and the exact workflow steps it should follow.

Good instructions are explicit about:
- What tools the agent has access to and when to use them
- The exact schema of the output (field names, types, valid values)
- What NOT to do (e.g. "do not assess risk — that is handled by later agents")

The instructions below are an abbreviated version of the production prompt. The full version is in `workflow_agent_prompts/threat_identifier.py`.

In [43]:
INSTRUCTIONS = """
You are an Expert Application Security Architect specialising in threat identification.

Your task: Identify security threats in the system architecture using the STRIDE methodology,
then write the results directly to the shared outputs/threats.json file.

You HAVE filesystem MCP access. You will:
- Read outputs/threats.json (current state: contains "metadata" and an empty "threats" array)
- Populate the "threats" array with threat objects
- Write the updated outputs/threats.json back

INPUTS PROVIDED (via the user message):
- Architecture diagram (mermaid format) showing system components and connections
- Business context describing what's critical, sensitive data, and compliance requirements

ANALYSIS STEPS:
1. First, produce a STRUCTURED ANALYSIS:
   - Assets: Identify the crown jewels — the most valuable data and systems
   - Entry Points: Logical parts of the architecture that provide mechanisms for external interaction
   - Trust Levels and Boundaries: Transitions between users, internet-facing services,
     internal services, data stores, third-party APIs, and administrative access paths
   - Attacker Profiles: Realistic threat actors (external attackers, malicious insiders,
     compromised supply chain, etc.) with their capabilities and motivations

2. Then, identify threats using STRIDE methodology:
   - Analyse each component against each STRIDE category
   - Consider the data flows and trust boundaries visible in the diagram
   - Cover every component and STRIDE category combination that is realistically applicable

STRIDE CATEGORIES:
  Spoofing | Tampering | Repudiation | Information Disclosure
  Denial of Service | Elevation of Privilege

THREAT GRAMMAR — every threat MUST follow this format:
  "[threat source] [prerequisite] can [threat action], which leads to [threat impact],
  resulting in reduced [impacted goal] of [impacted asset]."

WORKFLOW:
1. Read outputs/threats.json using the filesystem MCP read_file tool.
2. Parse the architecture diagram (mermaid) provided in the user message.
3. Analyse each component against each STRIDE category.
4. Identify a comprehensive set of realistic threats (typically 2-4 per critical component or data flow).
5. Populate the threats array with objects containing ONLY these 4 fields:
   - stride_category: one of the six STRIDE categories above
   - element: the affected component from the diagram
   - threat: the threat written in threat grammar format
   - attack_method: a full description of how the attack works
6. Write the updated outputs/threats.json back using write_file.

CRITICAL:
- Write valid JSON — no trailing commas, no comments
- Do NOT include risk, likelihood, or mitigation fields — later agents add those
- Never truncate or abbreviate content with "..."
- Every threat must be grounded in the architecture diagram — do not invent components
"""

## Creating an Agent

`Agent` is the core abstraction. It holds the instructions, the model, and any tools (including MCP servers).

`OpenAIChatCompletionsModel` is the key to using a custom API endpoint:
- `model` — the model name, passed as-is to the API (in our case the LiteLLM proxy routes it)
- `openai_client` — the `AsyncOpenAI` client you created above

This pattern lets you swap models or endpoints without touching the `Agent` definition.

In [44]:
agent = Agent(
    name="Threat Identifier Agent",
    instructions=INSTRUCTIONS,
    model=OpenAIChatCompletionsModel(model=MODEL, openai_client=client),
    mcp_servers=[],  # we will add the filesystem MCP server in a later step
)

## RunConfig and the Model Override

`RunConfig` is passed to `Runner.run()` to configure a specific run without modifying the `Agent` object itself.

> ⚠️ **Important**: If you pass a `model` to `RunConfig`, it **overrides** the model set on the `Agent`.

This is useful when:
- You want to keep the `Agent` definition generic and swap models at runtime
- You want to run the same agent with different models in tests vs production
- Multiple agents share a single `RunConfig` with the same client configuration

In the production pipeline, `run_config` is created once in `main.py` and passed to all four pipeline steps — so changing the model in one place affects all agents except the mitigation auditor which has its own run config. 

In [45]:
run_config = RunConfig(
    model=OpenAIChatCompletionsModel(model=MODEL, openai_client=client)
)

# To use a different model at runtime, simply change the model name here —
# the Agent's model definition is ignored when RunConfig.model is set:
#
# run_config = RunConfig(
#     model=OpenAIChatCompletionsModel(model="openai/gpt-4.1-mini", openai_client=client)
# )

## Running the Agent & Understanding "Tools"

`Runner.run()` executes the agent and manages the full **agentic loop**: LLM inference → tool calls → LLM inference → ... 
This loop repeats until the agent decides it has completed the task and stops calling tools.

### What is a Tool?
Think of a standard Large Language Model as a "brain in a jar." It is brilliant at reasoning and processing language, but it is completely isolated from the outside world. It can't browse the web, check the time, or modify files on your computer. 

**Tools give the agent hands.** A tool is a specific, secure capability or Python function that you expose to the LLM. Instead of trying to guess information, the agent can choose to pause, invoke a tool, read the real-world result, and use that data to figure out its next step.

### How the Model Context Protocol (MCP) Fits In
Instead of manually hardcoding custom tools for every single agent, we use the **Model Context Protocol (MCP)**. 

MCP is an open standard that lets you plug secure, pre-built servers directly into an agent. These MCP servers act as tool repositories. For example, in a few steps, we will plug in an **MCP Filesystem Server**. This server instantly provides the agent with a suite of secure tools like:
* `read_file`
* `write_file`
* `list_directory`

The agent uses these MCP-provided tools to safely step out of its "jar," look at your architecture files, and write its findings directly back to your project directory.

> 💡 **Safety Guardrail**: `max_turns=50` caps how many LLM calls can happen in a single run. This stops an agent from getting stuck in an infinite loop if it keeps calling a tool incorrectly over and over.



In [46]:
# Run a quick sanity check (no MCP server needed for this query):

# Ignore the [non-fatal] Tracing client error 401 about Incorrect API key

result = await Runner.run(
    agent,
    "List the six STRIDE threat categories and give a one-line description of each.",
    run_config=run_config,
    max_turns=5,
)
print(result.final_output)


Trace saved to outputs/traces/trace_threat_identifier.json
The six STRIDE threat categories are:

1. **Spoofing**: Unauthorized impersonation of a legitimate user or system, enabling the attacker to gain access to restricted resources.

2. **Tampering**: Unauthorized modification of data or systems, which can lead to incorrect information or system behavior.

3. **Repudiation**: Actions taken by users that can be denied later, resulting in a lack of accountability and potential disputes, often due to insufficient logging.

4. **Information Disclosure**: Unauthorized exposure of sensitive data to individuals or systems, leading to data breaches and privacy violations.

5. **Denial of Service (DoS)**: Disruption of service availability, preventing legitimate users from accessing resources, which can result from various forms of attack.

6. **Elevation of Privilege**: A user gains unauthorized access to higher-level permissions, allowing them to perform actions beyond their intended role

## Streaming Output

`Runner.run_streamed()` returns immediately with a stream object. You then iterate over `stream_events()` to receive chunks as they arrive.

This is what the production workflow uses — it lets you watch the agent's output in real time rather than waiting for the full response.

The `raw_response_event` events carry the streaming delta from the LLM. Other event types (tool calls, agent transitions, etc.) are also available but we ignore them here.

In [47]:
message = "Tell me a joke about threat modelling and STRIDE."

result = Runner.run_streamed(agent, message, run_config=run_config, max_turns=50)
async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, "delta"):
        print(event.data.delta, end="", flush=True)
print()  # newline after stream ends

Why did the threat modeler bring a ladder to the STRIDE meeting?

Because they wanted to elevate their privilege!
Trace saved to outputs/traces/trace_threat_identifier.json



## Adding an MCP Server (Filesystem Access)

The Threat Identifier agent needs to **read and write files** — specifically `outputs/threats.json` and `outputs/analysis.md`. Rather than giving it direct Python file access, we use the **Model Context Protocol (MCP) filesystem server**.

MCP servers expose tools to the agent (like `read_file`, `write_file`, `list_directory`) that the agent can call during its run. The filesystem MCP server is a Node.js process started by `npx`. For this to work, Node.js to be installed globally on your machine so that npx can dynamically pull and run the filesystem server. The code is configured to look for `npx` automatically in your system's global `PATH`. If you get a `FileNotFoundError`, make sure Node.js is installed correctly, or explicitly override the path by adding `NPX_PATH="/path/to/npx"` to your `.env` file.

`MCPServerStdio` is an **async context manager** — it starts the MCP process on `__aenter__` and shuts it down on `__aexit__`. The agent can only use its tools while inside the `async with` block.

In [48]:
# Ensure that when npx launches the MCP server, the child process can find node in the same nvm directory.
# This is needed only for Jupeter Notebook environments where the PATH may not include the nvm directory.
os.environ["PATH"] = (
    "/Users/stania.svobodova/.nvm/versions/node/v22.22.0/bin:"
    + os.environ.get("PATH", "")
)

In [49]:
FILESYSTEM_MCP_PARAMS = {
    "command": os.getenv("NPX_PATH", "npx"),
    "args": ["-y", "@modelcontextprotocol/server-filesystem", os.getcwd()],
}

message = "Read outputs/threats.json and count how many threats are already there."

async with MCPServerStdio(FILESYSTEM_MCP_PARAMS) as filesystem_mcp:
    agent = Agent(
        name="Threat Identifier Agent",
        instructions=INSTRUCTIONS,
        model=OpenAIChatCompletionsModel(model=MODEL, openai_client=client),
        mcp_servers=[filesystem_mcp],
    )
    result = Runner.run_streamed(agent, message, run_config=run_config, max_turns=50)
    async for event in result.stream_events():
        if event.type == "raw_response_event" and hasattr(event.data, "delta"):
            print(event.data.delta, end="", flush=True)
    print()

{"path":"outputs/threats.json"}The `outputs/threats.json` file currently contains no threats, as the "threats" array is empty.
Trace saved to outputs/traces/trace_threat_identifier.json



## 8. Tracing

The SDK has built-in tracing capabilities designed to capture every event during an agent run, including LLM generations, tool calls, and guardrail statuses. 

By default, these traces can be exported automatically to the OpenAI platform, where you can view and monitor them live on the [OpenAI Platform Dashboard](https://platform.openai.com/). While the OpenAI platform offers a beautiful, interactive user interface to visualize your agent's decision tree, sending data there comes with a compliance issue.

### The Security Dilemma: Local vs. Cloud Tracing
Traces generated by the Agents SDK don't just log basic runtime metrics; they record **everything**. This includes:
* Raw system instructions and user prompts.
* Full text-based inputs and outputs from every single tool execution.
* Intermediary thought patterns and the complete business logic of your application.

Because our workflow analyzes proprietary architecture diagrams and corporate security profiles, sending these traces to a third-party cloud platform would expose the **entire architectural structure and internal business logic of our platform**. To maintain strict data privacy, **we explicitly bypass cloud telemetry, saving traces only locally as JSON files.**

### Analyzing Local Traces with an IDE Agent
Since we forfeit OpenAI's visual UI to protect our data, parsing a massive, deeply nested local JSON trace file manually can be challenging. 

To solve this, we save the trace locally and pass it to an **IDE Agent** (such as Cursor, VS Code Copilot, or a local assistant) to do the heavy lifting. You can open the JSON trace file and ask the IDE agent targeted debugging questions, such as:
* *"Analyze this trace and pinpoint exactly why the agent failed on attempt 2."*
* *"Where did the agent make a mistake or deviate from the required JSON schema during its execution loop?"*

### Bypassing Initialization Check Errors
Even when we override the tracing system to keep files local, the OpenAI Agents SDK still scans the environment for an OpenAI key and will throw an error if it finds nothing. 

To prevent initialization failures while ensuring no live data leaks out to the cloud, set your `OPENAI_API_KEY` environment variable to a dummy string like `"unset"`. This satisfies the SDK's structural requirement while ensuring all actions remain contained safely within your local environment or custom LiteLLM proxy proxy.

```python
# Set this in your local environment or .env file to satisfy the SDK wrapper:
# OPENAI_API_KEY="unset"

In [50]:
class FileSpanExporter(TracingProcessor):
    """Writes all spans to a local JSON file when the trace ends."""

    def __init__(self, output_file: str = "trace_output.json"):
        self.output_file = output_file
        self._spans: list[dict] = []

    def on_trace_start(self, trace: Trace) -> None:
        self._spans = []

    def on_span_start(self, span: Span) -> None:
        pass

    def on_span_end(self, span: Span) -> None:
        span_export = span.export()
        if span_export is not None:
            self._spans.append(span_export)

    def on_trace_end(self, trace: Trace) -> None:
        output = {"trace": trace.export(), "spans": self._spans}
        with open(self.output_file, "w") as f:
            json.dump(output, f, indent=2, default=str)
        print(f"\nTrace saved to {self.output_file}")

    def shutdown(self) -> None:
        pass

    def force_flush(self) -> None:
        pass


# Before running the agent, redirect traces to a per-step file:
set_trace_processors([FileSpanExporter("outputs/traces/trace_threat_identifier.json")])

async with MCPServerStdio(FILESYSTEM_MCP_PARAMS) as filesystem_mcp:
    agent = Agent(
        name="Threat Identifier Agent",
        instructions=INSTRUCTIONS,
        model=OpenAIChatCompletionsModel(model=MODEL, openai_client=client),
        mcp_servers=[filesystem_mcp],
    )
    result = Runner.run_streamed(agent, message, run_config=run_config, max_turns=50)
    async for event in result.stream_events():
        if event.type == "raw_response_event" and hasattr(event.data, "delta"):
            print(event.data.delta, end="", flush=True)
    print()

{"path":"outputs/threats.json"}The `outputs/threats.json` file currently contains no threats, as the "threats" array is empty.
Trace saved to outputs/traces/trace_threat_identifier.json



Check the file inoutputs/traces/trace_threat_identifier.json to see how the trace looks like

## 9. Programmatic Validation

Instead of trusting the model to always produce correct output, we validate `outputs/threats.json` **after each agent run** in Python.

The validator returns:
- `None` — output is valid, proceed to the next step
- `str` — a description of all validation errors found

That error string is fed back to the agent on retry, so it knows exactly what to fix. This is more reliable than asking the agent to self-validate, because it runs outside the LLM context window and is deterministic.

In [51]:
VALID_STRIDE_CATEGORIES = {
    "Spoofing",
    "Tampering",
    "Repudiation",
    "Information Disclosure",
    "Denial of Service",
    "Elevation of Privilege",
}


def validate_after_threat_identifier(
    threat_count: int = 0,  # This is used for the next steps following threat identification, but we define it here so that all the valdate functions have the same signature
    threats_json_path: Path | None = None,
) -> str | None:
    """
    Validates outputs/threats.json after the Threat Identifier agent has run.
    Returns None if valid, or an error string listing every problem found.
    """
    path = threats_json_path or Path("outputs/threats.json")

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except FileNotFoundError:
        return "threats.json not found"
    except json.JSONDecodeError as e:
        return f"threats.json is not valid JSON: {e}"

    threats = data.get("threats", [])
    if not threats:
        return "threats array is empty — no threats were identified"

    errors: list[str] = []
    required_fields = {"stride_category", "element", "threat", "attack_method"}

    for i, threat in enumerate(threats):
        missing = required_fields - set(threat.keys())
        if missing:
            errors.append(f"Threat {i}: missing fields {missing}")
            continue

        for field in required_fields:
            val = threat.get(field)
            if not isinstance(val, str) or not val.strip():
                errors.append(
                    f"Threat {i}: field '{field}' must be a non-empty string, got: {repr(val)}"
                )

        if threat.get("stride_category") not in VALID_STRIDE_CATEGORIES:
            errors.append(
                f"Threat {i}: invalid stride_category '{threat.get('stride_category')}'"
            )

        # The threat identifier must NOT add fields belonging to later agents
        later_fields = {
            "impact",
            "likelihood",
            "risk",
            "all_possible_mitigations",
            "mitigations_already_in_place",
            "mitigations_missing",
            "ai_proposed_mitigations",
            "remaining_risk",
        }
        unexpected = {
            f for f in later_fields & set(threat.keys()) if threat.get(f) is not None
        }
        if unexpected:
            errors.append(f"Threat {i}: has unexpected non-null fields {unexpected}")

    return "\n".join(errors) if errors else None

In [ ]:
# no errors
# 0 - is just a placeholder for the next steps following threat identification, but we define it here so that all the valdate functions have the same signature
validation_result = validate_after_threat_identifier(
    0, Path("outputs/threats_after_threat_identifier.json")
)
print(validation_result)

None


In [55]:
# errors
validation_result = validate_after_threat_identifier(
    0, Path("outputs/threats_after_threat_identifier_with_errors.json")
)
print(validation_result)

Threat 6: invalid stride_category 'Goofing'
Threat 7: missing fields {'attack_method'}
Threat 11: has unexpected non-null fields {'likelihood'}


## 10. Validation + Retry Loop

The retry loop ties streaming, validation, and error feedback together:

1. Build the prompt (on retry, append the validation errors to the original message)
2. Run the agent with streaming
3. Validate the output
4. If valid → done. If not → retry with errors fed back as context

The agent sees its own mistakes as part of the next prompt, which is enough for most structural errors (wrong field names, missing fields, wrong STRIDE category values).

In [52]:
async def run_agent_with_validation(
    agent: Agent,
    input_message: str,
    validator,
    run_config: RunConfig,
    agent_name: str,
    threats_json_path: Path | None = None,
) -> None:
    """Run an agent with streaming, validate the output, and retry on failure."""
    path = threats_json_path or Path("outputs/threats.json")

    # Count threats before the run so the validator can detect truncation
    # This is used for the next steps following threat identification
    expected_threat_count = 0
    if path.exists():
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            expected_threat_count = len(data.get("threats", []))
        except json.JSONDecodeError, KeyError, TypeError:
            pass

    validation_error: str | None = None

    for attempt in range(1 + MAX_RETRIES):
        # On retry, append the validation errors to the original message so the
        # agent knows exactly what was wrong with its previous output
        if attempt > 0 and validation_error:
            message = (
                f"{input_message}\n\n"
                f"\u26a0\ufe0f VALIDATION FAILED (attempt {attempt}/{MAX_RETRIES}).\n"
                f"Errors:\n{validation_error}\n\n"
                f"Please re-read outputs/threats.json and fix these specific issues."
            )
        else:
            message = input_message

        print(f"  Running {agent_name} (attempt {attempt + 1})...")

        result = Runner.run_streamed(
            agent, message, run_config=run_config, max_turns=50
        )
        async for event in result.stream_events():
            if event.type == "raw_response_event" and hasattr(event.data, "delta"):
                print(event.data.delta, end="", flush=True)
        print()

        validation_error = validator(expected_threat_count, path)

        if validation_error is None:
            print(f"  \u2713 {agent_name} — validation passed")
            return

        print(f"  \u2717 {agent_name} — validation failed: {validation_error[:150]}")

    raise RuntimeError(
        f"{agent_name} FAILED after {1 + MAX_RETRIES} attempts.\n"
        f"Last errors: {validation_error}"
    )

## 11. The Full identify_threats() Step

This is the complete function, combining everything above. It matches the production implementation in `workflow_steps/threat_identification.py` exactly — the only difference is that we define it here inline instead of importing it.

Key points:
- The `Agent` is created **inside** the function so it is freshly initialised each run
- The `mcp_server` is passed in from the caller (which owns its lifecycle)
- `trace("Threat Identification")` groups all spans from this step in the trace output
- `run_agent_with_validation` handles streaming, validation, and retries

In [53]:
async def identify_threats(
    client: AsyncOpenAI,
    run_config: RunConfig,
    mcp_server,
    diagram: str,
    context: str,
    threats_json_path: Path | None = None,
) -> None:
    """Step 1: Identify STRIDE threats and write them to outputs/threats.json."""
    print("\n\U0001f4cb Step 1/4: Threat Identification")

    agent = Agent(
        name="Threat Identifier Agent",
        instructions=INSTRUCTIONS,
        model=OpenAIChatCompletionsModel(model=MODEL, openai_client=client),
        mcp_servers=[mcp_server],
    )

    agent_input = (
        f"Today's date: {TODAY}\n\n"
        f"Architecture diagram (mermaid):\n{diagram}\n\n"
        f"Business context:\n{context}"
    )

    # trace() groups all spans from this step into one named section in the trace file
    with trace("Threat Identification"):
        await run_agent_with_validation(
            agent,
            agent_input,
            validate_after_threat_identifier,
            run_config,
            "Threat Identifier",
            threats_json_path=threats_json_path,
        )

In [54]:
async with MCPServerStdio(FILESYSTEM_MCP_PARAMS) as filesystem_mcp:
    await identify_threats(
        client=client,
        run_config=run_config,
        mcp_server=filesystem_mcp,
        diagram=diagram,
        context=context,
    )


📋 Step 1/4: Threat Identification
  Running Threat Identifier (attempt 1)...
{"path":"outputs/threats.json"}### Structured Analysis

#### Assets
1. **API Gateway (APIGW)**: Public-facing entry point for API requests, requiring strict security.
2. **Lambda Function (Lambda)**: Processes user data and handles core business logic.
3. **DynamoDB Table (Dynamo)**: Stores critical application data, operational logs, and user submissions that are sensitive and must be secured.

#### Entry Points
- The API Gateway is the primary entry point for user interactions, allowing HTTPS requests from end-users.

#### Trust Levels and Boundaries
- **Trusted**: IAM roles managing interactions between authorized AWS services (Lambda, CloudWatch, DynamoDB).
- **Untrusted**: Incoming requests to API Gateway from the public internet, especially unauthenticated or malicious traffic.

#### Attacker Profiles
- **External Attackers**: Motivated by data theft or service disruption, possessing skills to exploit w

# BONUS: 

## Behind the Scenes: How the Agent Decides to Use a Tool

It can feel like magic when an agent decides to stop talking and suddenly opens a file. In reality, it is a highly structured game of matchmaking handled by the **OpenAI Agents SDK** and the LLM. 

The internal reasoning loop follows a four-step cycle: **Plan ➔ Choose ➔ Execute ➔ Reflect**.

### 1. The "Menu" Exposure (System Setup)
When you pass `mcp_servers=[filesystem_mcp]` to your `Agent`, the SDK dynamically fetches the definitions of all available tools from that server. It converts them into strict JSON schemas that describe:
* The **name** of the tool (e.g., `read_file`)
* A **description** of what it does (*"Reads the entire contents of a file..."*)
* The exact **parameters** required (*"path: a string representing the relative file path"*)

These descriptions are appended to your `INSTRUCTIONS` and sent to the LLM as part of its system context. 

### 2. The Decision to Call (The LLM "Thought")
When you ask the agent to *"Read outputs/threats.json and count the threats,"* the model reads your prompt and evaluates its own knowledge. It reasons:
> *"I do not know what is inside outputs/threats.json because I don't have that data in my training set. Let me look at my tool menu... Ah, there is a tool called `read_file` that takes a path string. I will call that."*

### 3. Intercepting the Tool Call (The SDK Action)
Instead of returning a standard text response like *"I am going to read the file now,"* the LLM outputs a structured **Tool Call** object:

```json
{
  "tool_name": "read_file",
  "arguments": { "path": "outputs/threats.json" }
}
```
The Runner.run() loop intercepts this structured block. It pauses the conversation, locates the filesystem MCP server, and physically executes the underlying Python/Node code to read your hard drive.

### 4. Feeding the Results Back (The Reflection)
Once the tool finishes running, the SDK takes the raw contents of threats.json and sends a new hidden message back to the LLM with a special role designation: role: tool.

The LLM wakes back up, reads this new data injection, and treats it as the absolute truth. It then continues its reasoning loop to answer your original prompt.

🧠 Key Takeaway for Design: Because the agent relies entirely on text descriptions to choose tools, semantic naming is everything. If you name a tool func_12b(), the agent will never use it. If you name it fetch_user_payment_history() and give it a crisp description, the LLM will match it perfectly whenever the user asks about billing.